# Alternative feature selection - 02 tuning recipe selection

This notebook consumes only the fold-plan and recipe-space artifacts written by `modeling_feature_selection_alternative_setup.ipynb`.

Within each assessment fold, this notebook fits prescreening and model recipes only on the corresponding training partition and scores them on tuning folds. It keeps validation score vectors only in compact in-memory blocks until the combined tuning-OOF metrics for the current assessment fold are computed. The notebook writes the small metric tables that later alternative-stage notebooks need.

## 0. Setup


In [11]:
from __future__ import annotations

import os

for _thread_env_var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ.setdefault(_thread_env_var, "1")

import json
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from cost_effective.dataset import find_project_root, load_training_data
from cost_effective.dataset.ensemble_feature_selection import collect_prescreen_rankings
from cost_effective.dataset.utils import DEFAULT_MAX_TARGETS
from cost_effective.models.feature_selection_alternative_inner_selection import (
    append_csv,
    build_fold_feature_candidates,
    evaluate_feature_candidates_oof_blocks,
    feature_cost_scale,
    filter_model_specs,
    filter_prescreen_recipes,
    inner_fold_indices,
    load_stage_one_tables,
    required_prescreen_methods,
    restrict_feature_candidates_to_selected_pairs,
    run_inner_fold_round_oof_blocks,
    score_inner_oof_prediction_blocks,
    select_top_feature_recipes,
    select_top_pipeline_recipes,
)
from cost_effective.notebook_artifacts import read_csv_if_available

project_root = find_project_root()
USE_EXISTING_OUTPUTS = True
FORCE_RERUN = False

stage_one_dir = (
    project_root / "outputs" / "feature_selection_alternative" / "01_fold_plan_and_recipe_space"
)
outputs = project_root / "outputs" / "feature_selection_alternative" / "02_inner_recipe_selection"
outputs.mkdir(parents=True, exist_ok=True)

required_stage_one_files = (
    "outer_fold_assignments.csv",
    "inner_fold_assignments.csv",
    "prescreen_recipe_space.csv",
    "feature_size_grid.csv",
    "model_spec_space.csv",
    "pipeline_recipe_space.csv",
    "leakage_contract.csv",
)
missing_stage_one_files = [
    name for name in required_stage_one_files if not (stage_one_dir / name).exists()
]
if missing_stage_one_files:
    raise FileNotFoundError(
        "Missing setup artifacts for the alternative feature-selection path: "
        f"{missing_stage_one_files}. Run "
        "notebooks/alternative_approach/modeling_feature_selection_alternative_setup.ipynb first."
    )

X_train, y_train = load_training_data(project_root / "data")
stage_one = load_stage_one_tables(stage_one_dir)

outer_assignments = stage_one["outer_assignments"]
inner_assignments = stage_one["inner_assignments"]
prescreen_recipes_all = stage_one["prescreen_recipes"]
feature_sizes_all = stage_one["feature_sizes"]
model_specs_all = stage_one["model_specs"]
leakage_contract = stage_one["leakage_contract"]

X_train.shape, y_train.shape, outer_assignments.shape, inner_assignments.shape

((5000, 500), (5000,), (5000, 2), (20000, 3))

In [12]:
RANDOM_STATE = 42
MAX_TARGETS = DEFAULT_MAX_TARGETS
FEATURE_COST_REFERENCE_ROW_COUNT = 1000
RANDOM_BASELINE_REPEATS = 20

DRY_RUN = False
OUTER_FOLDS_TO_RUN = None  # None means all outer folds.
INNER_FOLDS_TO_RUN = None  # None means all inner folds inside each outer fold.

INCLUDE_ADAPTIVE_WEIGHTED_PRESCREENS = True
MAX_PRESCREEN_RECIPES = None
FEATURE_SIZES_TO_RUN = None

ROUND1_MAX_FEATURE_CANDIDATES_PER_INNER = 300
ROUND1_SCREENING_MODEL_SPEC_IDS = (
    "extra_trees_small_deep02",
    "lightgbm_classifier_deep03",
    "xgboost_classifier_deep03",
    "random_forest_small_deep02",
    "logistic_baseline_deep03",
)
ROUND1_MAX_MODEL_SPECS = None

ROUND2_TOP_FEATURE_RECIPES_PER_OUTER = 60
ROUND2_MAX_MODEL_SPECS = None
SELECTED_PIPELINE_RECIPES_PER_OUTER = 10
MIN_INNER_FOLD_COUNT_ROUND1 = 2
MIN_INNER_FOLD_COUNT_ROUND2 = 5

# Metrics-only mode: train models, keep validation score vectors in memory per outer fold,
# and write only the combined OOF metric tables.
RUN_ROUND1_MODEL_FITS = bool(int(os.environ.get("ALTERNATIVE_FS_RUN_ROUND1_MODEL_FITS", "1")))
RUN_ROUND2_MODEL_FITS = bool(int(os.environ.get("ALTERNATIVE_FS_RUN_ROUND2_MODEL_FITS", "1")))
OVERWRITE_ROUND1_SCORES = bool(int(os.environ.get("ALTERNATIVE_FS_OVERWRITE_ROUND1_SCORES", "0")))
OVERWRITE_ROUND2_SCORES = bool(int(os.environ.get("ALTERNATIVE_FS_OVERWRITE_ROUND2_SCORES", "0")))
CLEAN_STAGE_OUTPUTS = False

LOG_EVERY_MODEL_FITS = 100
AVAILABLE_CPUS = (
    len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
)
DEFAULT_MODEL_FIT_N_JOBS = min(32, max(1, AVAILABLE_CPUS // 4))
MODEL_FIT_N_JOBS = int(os.environ.get("ALTERNATIVE_FS_MODEL_FIT_N_JOBS", DEFAULT_MODEL_FIT_N_JOBS))
PARALLEL_PREFER = os.environ.get("ALTERNATIVE_FS_PARALLEL_PREFER", "processes")
PARALLEL_VERBOSE = 0

if DRY_RUN:
    OUTER_FOLDS_TO_RUN = (1,)
    INNER_FOLDS_TO_RUN = (1, 2)
    MAX_PRESCREEN_RECIPES = 12
    FEATURE_SIZES_TO_RUN = (1, 2, 3, 5)
    ROUND1_MAX_FEATURE_CANDIDATES_PER_INNER = 20
    ROUND1_MAX_MODEL_SPECS = 2
    ROUND2_TOP_FEATURE_RECIPES_PER_OUTER = 5
    ROUND2_MAX_MODEL_SPECS = 3
    SELECTED_PIPELINE_RECIPES_PER_OUTER = 3
    MODEL_FIT_N_JOBS = 2
    MIN_INNER_FOLD_COUNT_ROUND1 = 1
    MIN_INNER_FOLD_COUNT_ROUND2 = 1
    RANDOM_BASELINE_REPEATS = 5

config = {
    "random_state": RANDOM_STATE,
    "max_targets": MAX_TARGETS,
    "feature_cost_reference_row_count": FEATURE_COST_REFERENCE_ROW_COUNT,
    "random_baseline_repeats": RANDOM_BASELINE_REPEATS,
    "scoring_formula": "10*TP - 5*FP - 200*n_features, scored once on combined tuning-OOF predictions per assessment fold",
    "storage_mode": "metrics_only_compact_oof_blocks_no_sample_prediction_csv",
    "dry_run": DRY_RUN,
    "assessment_folds_to_run": OUTER_FOLDS_TO_RUN,
    "tuning_folds_to_run": INNER_FOLDS_TO_RUN,
    "include_adaptive_weighted_prescreens": INCLUDE_ADAPTIVE_WEIGHTED_PRESCREENS,
    "max_prescreen_recipes": MAX_PRESCREEN_RECIPES,
    "feature_sizes_to_run": FEATURE_SIZES_TO_RUN,
    "round1_max_feature_candidates_per_inner": ROUND1_MAX_FEATURE_CANDIDATES_PER_INNER,
    "round1_screening_model_spec_ids": ROUND1_SCREENING_MODEL_SPEC_IDS,
    "round1_max_model_specs": ROUND1_MAX_MODEL_SPECS,
    "round2_top_feature_recipes_per_outer": ROUND2_TOP_FEATURE_RECIPES_PER_OUTER,
    "round2_max_model_specs": ROUND2_MAX_MODEL_SPECS,
    "selected_pipeline_recipes_per_outer": SELECTED_PIPELINE_RECIPES_PER_OUTER,
    "min_inner_fold_count_round1": MIN_INNER_FOLD_COUNT_ROUND1,
    "min_inner_fold_count_round2": MIN_INNER_FOLD_COUNT_ROUND2,
    "run_round1_model_fits": RUN_ROUND1_MODEL_FITS,
    "run_round2_model_fits": RUN_ROUND2_MODEL_FITS,
    "overwrite_round1_scores": OVERWRITE_ROUND1_SCORES,
    "overwrite_round2_scores": OVERWRITE_ROUND2_SCORES,
    "clean_stage_outputs": CLEAN_STAGE_OUTPUTS,
    "log_every_model_fits": LOG_EVERY_MODEL_FITS,
    "available_cpus": AVAILABLE_CPUS,
    "default_model_fit_n_jobs": DEFAULT_MODEL_FIT_N_JOBS,
    "model_fit_n_jobs": MODEL_FIT_N_JOBS,
    "parallel_prefer": PARALLEL_PREFER,
    "parallel_verbose": PARALLEL_VERBOSE,
    "required_previous_notebook": "notebooks/alternative_approach/modeling_feature_selection_alternative_setup.ipynb",
    "stage_one_dir": str(stage_one_dir.relative_to(project_root)),
    "output_dir": str(outputs.relative_to(project_root)),
}
with (outputs / "stage_config.json").open("w") as file_obj:
    json.dump(config, file_obj, indent=2, default=str)

pd.Series(config)

random_state                                                                              42
max_targets                                                                             1000
feature_cost_reference_row_count                                                        1000
random_baseline_repeats                                                                   20
scoring_formula                            10*TP - 5*FP - 200*n_features, scored once on ...
storage_mode                               metrics_only_compact_oof_blocks_no_sample_pred...
dry_run                                                                                False
assessment_folds_to_run                                                                 None
tuning_folds_to_run                                                                     None
include_adaptive_weighted_prescreens                                                    True
max_prescreen_recipes                                                 

## 1. Prepare executable recipe subsets from setup artifacts


In [13]:
outer_folds = sorted(outer_assignments["outer_fold"].unique())
if OUTER_FOLDS_TO_RUN is not None:
    outer_folds = [fold for fold in outer_folds if fold in set(OUTER_FOLDS_TO_RUN)]

feature_size_values = feature_sizes_all["feature_size"].astype(int).tolist()
if FEATURE_SIZES_TO_RUN is not None:
    feature_size_values = [
        size for size in feature_size_values if size in set(FEATURE_SIZES_TO_RUN)
    ]

prescreen_recipes = filter_prescreen_recipes(
    prescreen_recipes_all,
    include_adaptive=INCLUDE_ADAPTIVE_WEIGHTED_PRESCREENS,
    max_recipes=MAX_PRESCREEN_RECIPES,
)
round1_model_specs = filter_model_specs(
    model_specs_all,
    ROUND1_SCREENING_MODEL_SPEC_IDS,
    max_specs=ROUND1_MAX_MODEL_SPECS,
)
round2_model_specs = filter_model_specs(model_specs_all, max_specs=ROUND2_MAX_MODEL_SPECS)

execution_overview = pd.Series({
    "outer_folds": outer_folds,
    "feature_sizes": feature_size_values,
    "prescreen_recipes": len(prescreen_recipes),
    "round1_model_specs": len(round1_model_specs),
    "round2_model_specs": len(round2_model_specs),
})
execution_overview

outer_folds                                             [1, 2, 3, 4, 5]
feature_sizes         [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20, 30...
prescreen_recipes                                                   257
round1_model_specs                                                    5
round2_model_specs                                                   42
dtype: object

In [14]:
(
    round1_model_specs,
    round2_model_specs.groupby(["base_model_family", "model_kind"])
    .size()
    .rename("n_specs")
    .reset_index(),
)

(                model_spec_id    base_model_family  model_kind  \
 0    extra_trees_small_deep02    extra_trees_small  classifier   
 1  random_forest_small_deep02  random_forest_small  classifier   
 2  lightgbm_classifier_deep03  lightgbm_classifier  classifier   
 3    logistic_baseline_deep03    logistic_baseline  classifier   
 4   xgboost_classifier_deep03   xgboost_classifier  classifier   
 
                                         model_params  
 0  {"class_weight": "balanced", "max_depth": 4, "...  
 1  {"class_weight": "balanced_subsample", "max_de...  
 2  {"learning_rate": 0.035, "min_child_samples": ...  
 3  {"C": 0.2, "class_weight": "balanced", "l1_rat...  
 4  {"learning_rate": 0.035, "max_depth": 3, "min_...  ,
      base_model_family  model_kind  n_specs
 0         ebm_additive  classifier        3
 1    extra_trees_small  classifier        8
 2    lambdamart_ranker  lambdamart        3
 3  lightgbm_classifier  classifier        8
 4    logistic_baseline  classifie

## 2. Initialize logs and output paths


In [15]:
OUTPUT_FILES = {
    "run_log": outputs / "inner_cv_run_log.csv",
    "inner_cost_scale": outputs / "inner_validation_cost_scale.csv",
    "round1_oof_scores": outputs / "round1_inner_oof_scores.csv",
    "round1_oof_baselines": outputs / "round1_inner_oof_baselines.csv",
    "round1_method_failures": outputs / "round1_prescreen_method_failures.csv",
    "round1_recipe_failures": outputs / "round1_prescreen_recipe_failures.csv",
    "round1_selected_features": outputs / "round1_selected_feature_recipes.csv",
    "round2_oof_scores": outputs / "round2_inner_oof_scores.csv",
    "round2_oof_baselines": outputs / "round2_inner_oof_baselines.csv",
    "round2_method_failures": outputs / "round2_prescreen_method_failures.csv",
    "round2_recipe_failures": outputs / "round2_prescreen_recipe_failures.csv",
    "outer_selected_pipelines": outputs / "outer_selected_pipeline_recipes.csv",
    "stage_status_summary": outputs / "stage_status_summary.json",
    "output_manifest": outputs / "output_manifest.csv",
}

if CLEAN_STAGE_OUTPUTS:
    for path in OUTPUT_FILES.values():
        if path.exists():
            path.unlink()


def log_event(stage: str, message: str, **payload) -> None:
    row = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "stage": stage,
        "message": message,
        **payload,
    }
    append_csv(pd.DataFrame([row]), OUTPUT_FILES["run_log"])
    print(f"[{row['timestamp_utc']}] {stage}: {message} {payload}")


def output_has_outer_round(path: Path, outer_fold: int, round_name: str) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    columns = pd.read_csv(path, nrows=0).columns
    if not {"outer_fold", "round_name"}.issubset(columns):
        return False
    seen = pd.read_csv(path, usecols=["outer_fold", "round_name"])
    fold_rows = seen.loc[seen["outer_fold"].eq(int(outer_fold))]
    return bool(not fold_rows.empty and fold_rows["round_name"].eq(str(round_name)).any())


log_event(
    "setup",
    "initialized alternative stage-02 metrics-only outputs",
    output_dir=str(outputs.relative_to(project_root)),
    clean_stage_outputs=CLEAN_STAGE_OUTPUTS,
    storage_mode="metrics_only_compact_oof_blocks",
)

[2026-06-08T09:59:25.014029+00:00] setup: initialized alternative stage-02 metrics-only outputs {'output_dir': 'outputs/feature_selection_alternative/02_inner_recipe_selection', 'clean_stage_outputs': False, 'storage_mode': 'metrics_only_compact_oof_blocks'}


## 3. Tuning-fold size diagnostics


In [16]:
scale_rows = []
for outer_fold in outer_folds:
    inner_folds = sorted(
        inner_assignments.loc[inner_assignments["outer_fold"].eq(outer_fold), "inner_fold"].unique()
    )
    if INNER_FOLDS_TO_RUN is not None:
        inner_folds = [fold for fold in inner_folds if fold in set(INNER_FOLDS_TO_RUN)]
    for inner_fold in inner_folds:
        train_idx, val_idx = inner_fold_indices(inner_assignments, outer_fold, inner_fold)
        scale_rows.append({
            "outer_fold": int(outer_fold),
            "inner_fold": int(inner_fold),
            "inner_train_rows": len(train_idx),
            "inner_val_rows": len(val_idx),
            "feature_cost_reference_row_count": FEATURE_COST_REFERENCE_ROW_COUNT,
            "feature_cost_scale": feature_cost_scale(
                len(val_idx),
                reference_row_count=FEATURE_COST_REFERENCE_ROW_COUNT,
            ),
            "one_feature_scaled_cost": 200
            * feature_cost_scale(
                len(val_idx), reference_row_count=FEATURE_COST_REFERENCE_ROW_COUNT
            ),
        })

inner_validation_cost_scale = pd.DataFrame(scale_rows)
inner_validation_cost_scale.to_csv(OUTPUT_FILES["inner_cost_scale"], index=False)
log_event(
    "scoring_scale",
    "wrote inner validation cost scales",
    rows=len(inner_validation_cost_scale),
    min_scale=float(inner_validation_cost_scale["feature_cost_scale"].min()),
    max_scale=float(inner_validation_cost_scale["feature_cost_scale"].max()),
)
inner_validation_cost_scale.head(20)

[2026-06-08T09:59:27.580712+00:00] scoring_scale: wrote inner validation cost scales {'rows': 25, 'min_scale': 0.8, 'max_scale': 0.8}


,outer_fold,inner_fold,inner_train_rows,inner_val_rows,feature_cost_reference_row_count,feature_cost_scale,one_feature_scaled_cost
0,1,1,3200,800,1000,0.8,160.0
1,1,2,3200,800,1000,0.8,160.0
2,1,3,3200,800,1000,0.8,160.0
3,1,4,3200,800,1000,0.8,160.0
4,1,5,3200,800,1000,0.8,160.0
5,2,1,3200,800,1000,0.8,160.0
6,2,2,3200,800,1000,0.8,160.0
7,2,3,3200,800,1000,0.8,160.0
8,2,4,3200,800,1000,0.8,160.0
9,2,5,3200,800,1000,0.8,160.0


## 4. Round 1 - fold-local prescreen and feature-size screening

Round 1 trains screening models within each assessment fold training partition. It keeps tuning score vectors only until the combined tuning-OOF score for the current assessment fold is computed, then writes `round1_inner_oof_scores.csv` and `round1_inner_oof_baselines.csv`.


In [17]:
round1_round_name = "round1_prescreen_feature_screen"

if RUN_ROUND1_MODEL_FITS:
    if OVERWRITE_ROUND1_SCORES:
        for path in (OUTPUT_FILES["round1_oof_scores"], OUTPUT_FILES["round1_oof_baselines"]):
            if path.exists():
                path.unlink()

    for outer_fold in outer_folds:
        if output_has_outer_round(
            OUTPUT_FILES["round1_oof_scores"], int(outer_fold), round1_round_name
        ):
            log_event("round1_skip", "round 1 scores already exist", outer_fold=int(outer_fold))
            continue

        inner_folds = sorted(
            inner_assignments.loc[
                inner_assignments["outer_fold"].eq(outer_fold), "inner_fold"
            ].unique()
        )
        if INNER_FOLDS_TO_RUN is not None:
            inner_folds = [fold for fold in inner_folds if fold in set(INNER_FOLDS_TO_RUN)]

        outer_blocks = []
        for inner_fold in inner_folds:
            train_idx, val_idx = inner_fold_indices(inner_assignments, outer_fold, inner_fold)
            fold_scale = feature_cost_scale(
                len(val_idx),
                reference_row_count=FEATURE_COST_REFERENCE_ROW_COUNT,
            )
            log_event(
                "round1_start",
                "starting tuning fold round 1",
                outer_fold=int(outer_fold),
                inner_fold=int(inner_fold),
                inner_train_rows=len(train_idx),
                inner_val_rows=len(val_idx),
                feature_cost_scale=fold_scale,
                model_fit_n_jobs=MODEL_FIT_N_JOBS,
                parallel_prefer=PARALLEL_PREFER,
            )
            blocks, method_failures, recipe_failures = run_inner_fold_round_oof_blocks(
                X_train,
                y_train,
                train_idx,
                val_idx,
                prescreen_recipes,
                feature_size_values,
                round1_model_specs,
                outer_fold=int(outer_fold),
                inner_fold=int(inner_fold),
                round_name=round1_round_name,
                random_state=RANDOM_STATE,
                max_feature_candidates=ROUND1_MAX_FEATURE_CANDIDATES_PER_INNER,
                log_every=LOG_EVERY_MODEL_FITS,
                n_jobs=MODEL_FIT_N_JOBS,
                parallel_prefer=PARALLEL_PREFER,
                parallel_verbose=PARALLEL_VERBOSE,
            )
            outer_blocks.extend(blocks)
            if not method_failures.empty:
                append_csv(
                    method_failures.assign(
                        outer_fold=int(outer_fold),
                        inner_fold=int(inner_fold),
                        round_name=round1_round_name,
                    ),
                    OUTPUT_FILES["round1_method_failures"],
                )
            if not recipe_failures.empty:
                append_csv(
                    recipe_failures.assign(
                        outer_fold=int(outer_fold),
                        inner_fold=int(inner_fold),
                        round_name=round1_round_name,
                    ),
                    OUTPUT_FILES["round1_recipe_failures"],
                )
            log_event(
                "round1_fold_done",
                "finished tuning fold round 1 without writing predictions",
                outer_fold=int(outer_fold),
                inner_fold=int(inner_fold),
                compact_blocks=len(blocks),
                ok_blocks=sum(1 for block in blocks if block.get("status") == "ok"),
                method_failures=len(method_failures),
                recipe_failures=len(recipe_failures),
            )

        score_frame, baseline_frame = score_inner_oof_prediction_blocks(
            outer_blocks,
            max_targets=MAX_TARGETS,
            include_baseline_comparison=True,
            random_repeats=RANDOM_BASELINE_REPEATS,
            random_state=RANDOM_STATE + int(outer_fold),
            inner_assignments=inner_assignments,
            y=y_train,
            inner_folds=inner_folds,
        )
        append_csv(score_frame, OUTPUT_FILES["round1_oof_scores"])
        append_csv(baseline_frame, OUTPUT_FILES["round1_oof_baselines"])
        log_event(
            "round1_done",
            "scored round 1 combined tuning-OOF metrics",
            outer_fold=int(outer_fold),
            score_rows=len(score_frame),
            baseline_rows=len(baseline_frame),
            ok_score_rows=int(score_frame["status"].eq("ok").sum()) if not score_frame.empty else 0,
        )
else:
    log_event(
        "round1_recovery",
        "round 1 model fitting disabled; loading existing alternative score or selection CSV",
    )

round1_inner_oof_scores = read_csv_if_available(OUTPUT_FILES["round1_oof_scores"])
round1_inner_oof_baselines = read_csv_if_available(OUTPUT_FILES["round1_oof_baselines"])
existing_round1_selected_feature_recipes = read_csv_if_available(
    OUTPUT_FILES["round1_selected_features"]
)
round1_scores_are_compact_oof = "inner_oof_business_score" in round1_inner_oof_scores.columns
round1_scores_are_legacy_inner_fold = (
    "inner_business_score" in round1_inner_oof_scores.columns and not round1_scores_are_compact_oof
)

if round1_inner_oof_scores.empty and existing_round1_selected_feature_recipes.empty:
    raise FileNotFoundError(
        "round1_inner_oof_scores.csv and round1_selected_feature_recipes.csv are both missing. "
        "Set RUN_ROUND1_MODEL_FITS=True or restore the round-1 selected feature artifact."
    )
if not round1_inner_oof_scores.empty and not round1_scores_are_compact_oof:
    if existing_round1_selected_feature_recipes.empty:
        raise FileNotFoundError(
            "round1_inner_oof_scores.csv uses the older per-inner-fold score format and "
            "round1_selected_feature_recipes.csv is missing. Restore the selected-feature "
            "artifact or regenerate round-1 compact OOF scores."
        )
    log_event(
        "round1_recovery",
        "round 1 scores use the older per-inner-fold format; using existing selected feature recipes for round 2",
        score_format="legacy_inner_fold"
        if round1_scores_are_legacy_inner_fold
        else "unknown_non_compact",
        selected_feature_rows=len(existing_round1_selected_feature_recipes),
    )
elif round1_inner_oof_scores.empty:
    log_event(
        "round1_recovery",
        "round 1 OOF scores are unavailable; using existing alternative selected feature recipes for round 2",
        selected_feature_rows=len(existing_round1_selected_feature_recipes),
    )
round1_inner_oof_scores.shape, round1_inner_oof_scores.head()

[2026-06-08T09:59:31.369002+00:00] round1_skip: round 1 scores already exist {'outer_fold': 1}
[2026-06-08T09:59:31.903885+00:00] round1_skip: round 1 scores already exist {'outer_fold': 2}
[2026-06-08T09:59:32.433869+00:00] round1_skip: round 1 scores already exist {'outer_fold': 3}
[2026-06-08T09:59:32.948479+00:00] round1_skip: round 1 scores already exist {'outer_fold': 4}
[2026-06-08T09:59:33.493744+00:00] round1_skip: round 1 scores already exist {'outer_fold': 5}
[2026-06-08T09:59:34.264992+00:00] round1_recovery: round 1 scores use the older per-inner-fold format; using existing selected feature recipes for round 2 {'score_format': 'legacy_inner_fold', 'selected_feature_rows': 300}


((37500, 37),
    outer_fold  inner_fold                       round_name  \
 0           1           1  round1_prescreen_feature_screen   
 1           1           1  round1_prescreen_feature_screen   
 2           1           1  round1_prescreen_feature_screen   
 3           1           1  round1_prescreen_feature_screen   
 4           1           1  round1_prescreen_feature_screen   
 
                                 pipeline_recipe_id    feature_recipe_id  \
 0    ps_mean_0001__k_001__extra_trees_small_deep02  ps_mean_0001__k_001   
 1  ps_mean_0001__k_001__random_forest_small_deep02  ps_mean_0001__k_001   
 2  ps_mean_0001__k_001__lightgbm_classifier_deep03  ps_mean_0001__k_001   
 3    ps_mean_0001__k_001__logistic_baseline_deep03  ps_mean_0001__k_001   
 4   ps_mean_0001__k_001__xgboost_classifier_deep03  ps_mean_0001__k_001   
 
   prescreen_recipe_id       prescreen_name prescreen_methods rank_aggregation  \
 0        ps_mean_0001  single__mutual_info   ["mutual_info"]  mea

## 5. Select feature recipes per outer fold


In [18]:
if not existing_round1_selected_feature_recipes.empty and (
    round1_inner_oof_scores.empty or not round1_scores_are_compact_oof
):
    round1_selected_feature_recipes = existing_round1_selected_feature_recipes.copy()
    log_event(
        "round1_selection_recovery",
        "loaded existing round-1 selected feature recipes",
        selected_feature_recipes=len(round1_selected_feature_recipes),
        reason="missing_or_legacy_round1_oof_scores",
    )
else:
    round1_selected_frames = []
    for outer_fold in outer_folds:
        fold_scores = round1_inner_oof_scores.loc[
            round1_inner_oof_scores["outer_fold"].eq(outer_fold)
        ].copy()
        selected = select_top_feature_recipes(
            fold_scores,
            top_n=ROUND2_TOP_FEATURE_RECIPES_PER_OUTER,
            min_inner_fold_count=MIN_INNER_FOLD_COUNT_ROUND1,
        )
        if not selected.empty:
            selected["outer_fold"] = int(outer_fold)
            round1_selected_frames.append(selected)
            log_event(
                "round1_selection",
                "selected feature recipes for round 2 by combined tuning-OOF score",
                outer_fold=int(outer_fold),
                selected_feature_recipes=len(selected),
                best_inner_oof_business_score=float(selected.iloc[0]["inner_oof_business_score"]),
            )

    round1_selected_feature_recipes = (
        pd.concat(round1_selected_frames, ignore_index=True)
        if round1_selected_frames
        else pd.DataFrame()
    )
    round1_selected_feature_recipes.to_csv(OUTPUT_FILES["round1_selected_features"], index=False)

round1_selected_feature_recipes.head(30)

[2026-06-08T09:59:37.057553+00:00] round1_selection_recovery: loaded existing round-1 selected feature recipes {'selected_feature_recipes': 300, 'reason': 'missing_or_legacy_round1_oof_scores'}


,outer_fold,round_name,prescreen_recipe_id,inner_oof_business_score,inner_oof_optimal_k,inner_oof_f1_score,inner_oof_roc_auc_score,inner_oof_n_predictions,inner_oof_selected_rate,selected_features_text,...,inner_oof_fp_at_optimal_k,baseline_select_all_capped_features_0_business_score,baseline_select_all_capped_features_1_business_score,baseline_select_all_illegal_full_features_0_business_score,baseline_select_all_illegal_full_features_1_business_score,beats_baseline_select_all_capped_features_0,beats_baseline_select_all_capped_features_1,beats_baseline_select_all_illegal_full_features_0,beats_baseline_select_all_illegal_full_features_1,inner_selection_rank
0,1,round1_prescreen_feature_screen,ps_mean_0005,4458.0,998.0,0.483884,0.684578,4000,0.24950,"var_159,var_175,var_190,var_214,var_254,var_34...",...,274.8,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,1
1,1,round1_prescreen_feature_screen,ps_mean_0007,4456.0,995.8,0.474986,0.668632,4000,0.24895,"var_175,var_190,var_214,var_254,var_341,var_379",...,286.8,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,2
2,1,round1_prescreen_feature_screen,ps_mean_0007,4408.0,996.2,0.490573,0.698772,4000,0.24905,"var_175,var_190,var_214,var_254,var_341,var_37...",...,263.6,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,3
3,1,round1_prescreen_feature_screen,ps_mean_0007,4405.0,999.6,0.481641,0.683676,4000,0.24990,"var_175,var_190,var_214,var_254,var_341,var_37...",...,279.4,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,4
4,1,round1_prescreen_feature_screen,ps_mean_0005,4396.0,999.2,0.490139,0.696342,4000,0.24980,"var_116,var_159,var_175,var_190,var_214,var_25...",...,266.4,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,5
5,1,round1_prescreen_feature_screen,ps_mean_0005,4379.0,999.6,0.462652,0.652172,4000,0.24990,"var_190,var_214,var_254,var_341,var_379",...,307.8,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,6
6,1,round1_prescreen_feature_screen,ps_mean_0018,4323.0,998.0,0.451209,0.634072,4000,0.24950,"var_10,var_175,var_254,var_389",...,323.8,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,7
7,1,round1_prescreen_feature_screen,ps_mean_0020,4312.0,995.0,0.459558,0.650644,4000,0.24875,"var_175,var_223,var_254,var_328,var_389",...,309.2,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,8
8,1,round1_prescreen_feature_screen,ps_mean_0007,4294.0,998.0,0.458874,0.648046,4000,0.24950,"var_175,var_190,var_214,var_254,var_341",...,312.4,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,9
9,1,round1_prescreen_feature_screen,ps_mean_0005,4266.0,997.8,0.466529,0.661690,4000,0.24945,"var_159,var_190,var_214,var_254,var_341,var_379",...,300.8,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,10


## 6. Round 2 - full model/HPO selection inside each assessment fold training partition

Round 2 also uses metrics-only compact blocks. It does not append a sample-level prediction CSV; after each assessment fold is completed, the combined tuning-OOF metrics are written directly to `round2_inner_oof_scores.csv`.


In [19]:
round2_round_name = "round2_model_hpo_selection"

if RUN_ROUND2_MODEL_FITS:
    if OVERWRITE_ROUND2_SCORES:
        for path in (OUTPUT_FILES["round2_oof_scores"], OUTPUT_FILES["round2_oof_baselines"]):
            if path.exists():
                path.unlink()

    for outer_fold in outer_folds:
        if output_has_outer_round(
            OUTPUT_FILES["round2_oof_scores"], int(outer_fold), round2_round_name
        ):
            log_event("round2_skip", "round 2 scores already exist", outer_fold=int(outer_fold))
            continue

        selected_pairs = round1_selected_feature_recipes.loc[
            round1_selected_feature_recipes["outer_fold"].eq(outer_fold)
        ].copy()
        if selected_pairs.empty:
            log_event("round2_skip", "no selected feature recipes", outer_fold=int(outer_fold))
            continue

        selected_prescreen_ids = selected_pairs["prescreen_recipe_id"].drop_duplicates().tolist()
        selected_sizes = sorted(
            selected_pairs["feature_size"].astype(int).drop_duplicates().tolist()
        )
        selected_prescreens = prescreen_recipes.loc[
            prescreen_recipes["prescreen_recipe_id"].isin(selected_prescreen_ids)
        ].copy()

        inner_folds = sorted(
            inner_assignments.loc[
                inner_assignments["outer_fold"].eq(outer_fold), "inner_fold"
            ].unique()
        )
        if INNER_FOLDS_TO_RUN is not None:
            inner_folds = [fold for fold in inner_folds if fold in set(INNER_FOLDS_TO_RUN)]

        outer_blocks = []
        for inner_fold in inner_folds:
            train_idx, val_idx = inner_fold_indices(inner_assignments, outer_fold, inner_fold)
            x_inner_train = X_train.iloc[list(train_idx)]
            y_inner_train = y_train.iloc[list(train_idx)]
            x_inner_val = X_train.iloc[list(val_idx)]
            y_inner_val = y_train.iloc[list(val_idx)]
            fold_scale = feature_cost_scale(
                len(val_idx),
                reference_row_count=FEATURE_COST_REFERENCE_ROW_COUNT,
            )
            log_event(
                "round2_start",
                "starting tuning fold round 2",
                outer_fold=int(outer_fold),
                inner_fold=int(inner_fold),
                inner_train_rows=len(train_idx),
                inner_val_rows=len(val_idx),
                feature_cost_scale=fold_scale,
                selected_prescreens=len(selected_prescreens),
                selected_feature_sizes=len(selected_sizes),
                model_specs=len(round2_model_specs),
                model_fit_n_jobs=MODEL_FIT_N_JOBS,
                parallel_prefer=PARALLEL_PREFER,
            )

            methods = required_prescreen_methods(selected_prescreens)
            rankings, method_failures = collect_prescreen_rankings(
                x_inner_train,
                y_inner_train,
                methods=methods,
                random_state=RANDOM_STATE + int(outer_fold) * 10000 + int(inner_fold) * 100,
                continue_on_error=True,
            )
            feature_candidates, recipe_failures = build_fold_feature_candidates(
                rankings,
                selected_prescreens,
                selected_sizes,
                max_feature_candidates=None,
            )
            feature_candidates = restrict_feature_candidates_to_selected_pairs(
                feature_candidates,
                selected_pairs,
            )
            blocks = evaluate_feature_candidates_oof_blocks(
                x_inner_train,
                y_inner_train,
                x_inner_val,
                y_inner_val,
                feature_candidates,
                round2_model_specs,
                outer_fold=int(outer_fold),
                inner_fold=int(inner_fold),
                round_name=round2_round_name,
                random_state=RANDOM_STATE,
                log_every=LOG_EVERY_MODEL_FITS,
                n_jobs=MODEL_FIT_N_JOBS,
                parallel_prefer=PARALLEL_PREFER,
                parallel_verbose=PARALLEL_VERBOSE,
            )
            outer_blocks.extend(blocks)
            if not method_failures.empty:
                append_csv(
                    method_failures.assign(
                        outer_fold=int(outer_fold),
                        inner_fold=int(inner_fold),
                        round_name=round2_round_name,
                    ),
                    OUTPUT_FILES["round2_method_failures"],
                )
            if not recipe_failures.empty:
                append_csv(
                    recipe_failures.assign(
                        outer_fold=int(outer_fold),
                        inner_fold=int(inner_fold),
                        round_name=round2_round_name,
                    ),
                    OUTPUT_FILES["round2_recipe_failures"],
                )
            log_event(
                "round2_fold_done",
                "finished tuning fold round 2 without writing predictions",
                outer_fold=int(outer_fold),
                inner_fold=int(inner_fold),
                feature_candidates=len(feature_candidates),
                compact_blocks=len(blocks),
                ok_blocks=sum(1 for block in blocks if block.get("status") == "ok"),
                method_failures=len(method_failures),
                recipe_failures=len(recipe_failures),
            )

        score_frame, baseline_frame = score_inner_oof_prediction_blocks(
            outer_blocks,
            max_targets=MAX_TARGETS,
            include_baseline_comparison=True,
            random_repeats=RANDOM_BASELINE_REPEATS,
            random_state=RANDOM_STATE + int(outer_fold),
            inner_assignments=inner_assignments,
            y=y_train,
            inner_folds=inner_folds,
        )
        append_csv(score_frame, OUTPUT_FILES["round2_oof_scores"])
        append_csv(baseline_frame, OUTPUT_FILES["round2_oof_baselines"])
        log_event(
            "round2_done",
            "scored round 2 combined tuning-OOF metrics",
            outer_fold=int(outer_fold),
            score_rows=len(score_frame),
            baseline_rows=len(baseline_frame),
            ok_score_rows=int(score_frame["status"].eq("ok").sum()) if not score_frame.empty else 0,
        )
else:
    log_event(
        "round2_recovery",
        "round 2 model fitting disabled; loading existing alternative OOF score CSV",
    )

round2_inner_oof_scores = read_csv_if_available(OUTPUT_FILES["round2_oof_scores"])
round2_inner_oof_baselines = read_csv_if_available(OUTPUT_FILES["round2_oof_baselines"])
if round2_inner_oof_scores.empty:
    raise FileNotFoundError(
        "round2_inner_oof_scores.csv is missing or empty. This notebook no longer uses "
        "the sample-level prediction CSV; set RUN_ROUND2_MODEL_FITS=True to regenerate compact OOF metrics."
    )
round2_inner_oof_scores.shape, round2_inner_oof_scores.head()

[2026-06-08T09:59:41.820293+00:00] round2_skip: round 2 scores already exist {'outer_fold': 1}
[2026-06-08T09:59:41.960860+00:00] round2_skip: round 2 scores already exist {'outer_fold': 2}
[2026-06-08T09:59:42.112071+00:00] round2_skip: round 2 scores already exist {'outer_fold': 3}
[2026-06-08T09:59:42.288799+00:00] round2_skip: round 2 scores already exist {'outer_fold': 4}
[2026-06-08T09:59:42.481939+00:00] round2_skip: round 2 scores already exist {'outer_fold': 5}


((12600, 43),
    outer_fold                  round_name  \
 0           1  round2_model_hpo_selection   
 1           1  round2_model_hpo_selection   
 2           1  round2_model_hpo_selection   
 3           1  round2_model_hpo_selection   
 4           1  round2_model_hpo_selection   
 
                               pipeline_recipe_id    feature_recipe_id  \
 0  ps_mean_0007__k_008__extra_trees_small_deep02  ps_mean_0007__k_008   
 1  ps_mean_0007__k_008__extra_trees_small_deep04  ps_mean_0007__k_008   
 2  ps_mean_0007__k_007__extra_trees_small_deep04  ps_mean_0007__k_007   
 3       ps_mean_0005__k_007__ebm_additive_deep02  ps_mean_0005__k_007   
 4  ps_mean_0007__k_007__extra_trees_small_deep08  ps_mean_0007__k_007   
 
   prescreen_recipe_id           prescreen_name    prescreen_methods  \
 0        ps_mean_0007  single__sparse_gam_spam  ["sparse_gam_spam"]   
 1        ps_mean_0007  single__sparse_gam_spam  ["sparse_gam_spam"]   
 2        ps_mean_0007  single__sparse_gam_spa

## 7. Select full pipeline recipes per assessment fold


In [20]:
selected_pipeline_frames = []
for outer_fold in outer_folds:
    fold_scores = round2_inner_oof_scores.loc[
        round2_inner_oof_scores["outer_fold"].eq(outer_fold)
    ].copy()
    selected = select_top_pipeline_recipes(
        fold_scores,
        top_n=SELECTED_PIPELINE_RECIPES_PER_OUTER,
        min_inner_fold_count=MIN_INNER_FOLD_COUNT_ROUND2,
    )
    if not selected.empty:
        selected["outer_fold"] = int(outer_fold)
        selected_pipeline_frames.append(selected)
        log_event(
            "round2_selection",
            "selected full pipeline recipes for alternative evaluation by combined tuning-OOF score",
            outer_fold=int(outer_fold),
            selected_pipeline_recipes=len(selected),
            best_inner_oof_business_score=float(selected.iloc[0]["inner_oof_business_score"]),
        )

outer_selected_pipeline_recipes = (
    pd.concat(selected_pipeline_frames, ignore_index=True)
    if selected_pipeline_frames
    else pd.DataFrame()
)
outer_selected_pipeline_recipes.to_csv(OUTPUT_FILES["outer_selected_pipelines"], index=False)
outer_selected_pipeline_recipes.head(50)

[2026-06-08T09:59:48.763981+00:00] round2_selection: selected full pipeline recipes for alternative evaluation by combined tuning-OOF score {'outer_fold': 1, 'selected_pipeline_recipes': 10, 'best_inner_oof_business_score': 5190.0}
[2026-06-08T09:59:48.825393+00:00] round2_selection: selected full pipeline recipes for alternative evaluation by combined tuning-OOF score {'outer_fold': 2, 'selected_pipeline_recipes': 10, 'best_inner_oof_business_score': 4995.0}
[2026-06-08T09:59:48.879438+00:00] round2_selection: selected full pipeline recipes for alternative evaluation by combined tuning-OOF score {'outer_fold': 3, 'selected_pipeline_recipes': 10, 'best_inner_oof_business_score': 5035.0}
[2026-06-08T09:59:48.935574+00:00] round2_selection: selected full pipeline recipes for alternative evaluation by combined tuning-OOF score {'outer_fold': 4, 'selected_pipeline_recipes': 10, 'best_inner_oof_business_score': 5165.0}
[2026-06-08T09:59:48.992470+00:00] round2_selection: selected full pipel

,outer_fold,round_name,pipeline_recipe_id,inner_oof_business_score,inner_oof_optimal_k,inner_oof_f1_score,inner_oof_roc_auc_score,inner_oof_n_predictions,inner_oof_selected_rate,selected_features_text,...,inner_oof_fp_at_optimal_k,baseline_select_all_capped_features_0_business_score,baseline_select_all_capped_features_1_business_score,baseline_select_all_illegal_full_features_0_business_score,baseline_select_all_illegal_full_features_1_business_score,beats_baseline_select_all_capped_features_0,beats_baseline_select_all_capped_features_1,beats_baseline_select_all_illegal_full_features_0,beats_baseline_select_all_illegal_full_features_1,inner_selection_rank
0,1,round2_model_hpo_selection,ps_mean_0007__k_008__extra_trees_small_deep02,5190.0,1000.0,0.525577,0.732854,4000,0.25000,"var_175,var_190,var_214,var_254,var_341,var_37...",...,214.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,1
1,1,round2_model_hpo_selection,ps_mean_0007__k_008__extra_trees_small_deep04,5160.0,1000.0,0.524239,0.735960,4000,0.25000,"var_175,var_190,var_214,var_254,var_341,var_37...",...,216.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,2
2,1,round2_model_hpo_selection,ps_mean_0007__k_007__extra_trees_small_deep04,5120.0,1000.0,0.513541,0.714783,4000,0.25000,"var_175,var_190,var_214,var_254,var_341,var_37...",...,232.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,3
3,1,round2_model_hpo_selection,ps_mean_0005__k_007__ebm_additive_deep02,5115.0,998.0,0.513215,0.725130,4000,0.24950,"var_159,var_175,var_190,var_214,var_254,var_34...",...,231.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,4
4,1,round2_model_hpo_selection,ps_mean_0007__k_007__extra_trees_small_deep08,5110.0,999.0,0.513043,0.714498,4000,0.24975,"var_175,var_190,var_214,var_254,var_341,var_37...",...,232.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,5
5,1,round2_model_hpo_selection,ps_mean_0007__k_008__ebm_additive_deep02,5105.0,999.0,0.521739,0.735097,4000,0.24975,"var_175,var_190,var_214,var_254,var_341,var_37...",...,219.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,6
6,1,round2_model_hpo_selection,ps_mean_0005__k_007__extra_trees_small_deep01,5105.0,1000.0,0.512872,0.717405,4000,0.25000,"var_159,var_175,var_190,var_214,var_254,var_34...",...,233.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,7
7,1,round2_model_hpo_selection,ps_mean_0005__k_008__ebm_additive_deep03,5100.0,991.0,0.521127,0.739561,4000,0.24775,"var_116,var_159,var_175,var_190,var_214,var_25...",...,214.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,8
8,1,round2_model_hpo_selection,ps_mean_0007__k_007__extra_trees_small_deep03,5090.0,1000.0,0.512203,0.716404,4000,0.25000,"var_175,var_190,var_214,var_254,var_341,var_37...",...,234.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,9
9,1,round2_model_hpo_selection,ps_mean_0005__k_008__ebm_additive_deep02,5075.0,999.0,0.520401,0.739568,4000,0.24975,"var_116,var_159,var_175,var_190,var_214,var_25...",...,221.0,2200.0,2000.0,9865.0,9665.0,1.0,1.0,0.0,0.0,10


## 8. Status summaries


In [21]:
status_summary = {
    "scoring_version": "combined_inner_oof_business_score_v1",
    "storage_mode": "metrics_only_compact_oof_blocks_no_sample_prediction_csv",
    "business_objective": "10*TP - 5*FP - 200*n_features",
    "max_targets": MAX_TARGETS,
    "feature_cost_reference_row_count": FEATURE_COST_REFERENCE_ROW_COUNT,
    "random_baseline_repeats": RANDOM_BASELINE_REPEATS,
    "run_round1_model_fits": RUN_ROUND1_MODEL_FITS,
    "run_round2_model_fits": RUN_ROUND2_MODEL_FITS,
    "round1_inner_oof_scores": round1_inner_oof_scores["status"]
    .value_counts(dropna=False)
    .to_dict()
    if not round1_inner_oof_scores.empty
    else {},
    "round1_selected_feature_rows": len(round1_selected_feature_recipes),
    "round1_baselines": round1_inner_oof_baselines["baseline_type"]
    .value_counts(dropna=False)
    .to_dict()
    if not round1_inner_oof_baselines.empty
    else {},
    "round2_inner_oof_scores": round2_inner_oof_scores["status"]
    .value_counts(dropna=False)
    .to_dict()
    if not round2_inner_oof_scores.empty
    else {},
    "round2_baselines": round2_inner_oof_baselines["baseline_type"]
    .value_counts(dropna=False)
    .to_dict()
    if not round2_inner_oof_baselines.empty
    else {},
    "outer_selected_pipeline_rows": len(outer_selected_pipeline_recipes),
    "sample_level_prediction_csv_written": False,
}
with OUTPUT_FILES["stage_status_summary"].open("w") as file_obj:
    json.dump(status_summary, file_obj, indent=2, default=str)

pd.Series(status_summary)

scoring_version                                     combined_inner_oof_business_score_v1
storage_mode                           metrics_only_compact_oof_blocks_no_sample_pred...
business_objective                                         10*TP - 5*FP - 200*n_features
max_targets                                                                         1000
feature_cost_reference_row_count                                                    1000
random_baseline_repeats                                                               20
run_round1_model_fits                                                               True
run_round2_model_fits                                                               True
round1_inner_oof_scores                                                    {'ok': 37500}
round1_selected_feature_rows                                                         300
round1_baselines                                                                      {}
round2_inner_oof_scor

In [ ]:
if not outer_selected_pipeline_recipes.empty:
    display_cols = [
        "outer_fold",
        "inner_selection_rank",
        "pipeline_recipe_id",
        "prescreen_recipe_id",
        "prescreen_name",
        "feature_size",
        "model_spec_id",
        "base_model_family",
        "inner_oof_business_score",
        "inner_oof_optimal_k",
        "inner_oof_selected_rate",
        "inner_oof_f1_score",
        "inner_oof_roc_auc_score",
        "inner_oof_n_predictions",
        "feature_penalty",
        "gross_score",
        "baseline_select_all_capped_features_0_business_score",
        "baseline_select_all_capped_features_1_business_score",
        "baseline_select_all_illegal_full_features_0_business_score",
        "baseline_select_all_illegal_full_features_1_business_score",
        "beats_baseline_select_all_capped_features_0",
        "beats_baseline_select_all_capped_features_1",
        "beats_baseline_select_all_illegal_full_features_0",
        "beats_baseline_select_all_illegal_full_features_1",
        "missing_inner_oof_predictions",
        "missing_inner_fold_count",
        "manifest_checked",
        "inner_fold_count",
    ]
    display_cols = [col for col in display_cols if col in outer_selected_pipeline_recipes.columns]
    outer_selected_pipeline_recipes[display_cols].head(50)
else:
    display(outer_selected_pipeline_recipes)

## 9. Output manifest


In [23]:
output_manifest = pd.DataFrame([
    {"file": "stage_config.json", "meaning": "Alternative stage-02 execution configuration."},
    {"file": "inner_cv_run_log.csv", "meaning": "Timestamped progress log."},
    {"file": "inner_validation_cost_scale.csv", "meaning": "Per-inner-fold row-count diagnostic."},
    {
        "file": "round1_inner_oof_scores.csv",
        "meaning": "Round 1 recipe scores computed directly from compact in-memory tuning-OOF blocks.",
    },
    {
        "file": "round1_inner_oof_baselines.csv",
        "meaning": "Round 1 select-all and random baseline diagnostics.",
    },
    {
        "file": "round1_selected_feature_recipes.csv",
        "meaning": "Feature recipes selected per outer fold for round 2; can be reused if round-1 OOF scores are unavailable.",
    },
    {
        "file": "round2_inner_oof_scores.csv",
        "meaning": "Round 2 full pipeline scores computed directly from compact in-memory tuning-OOF blocks.",
    },
    {
        "file": "round2_inner_oof_baselines.csv",
        "meaning": "Round 2 select-all and random baseline diagnostics.",
    },
    {
        "file": "outer_selected_pipeline_recipes.csv",
        "meaning": "Final per-assessment-fold recipes for alternative evaluation.",
    },
    {
        "file": "round1_prescreen_method_failures.csv",
        "meaning": "Optional method failures in round 1, if any.",
    },
    {
        "file": "round1_prescreen_recipe_failures.csv",
        "meaning": "Skipped prescreen recipes in round 1, if any.",
    },
    {
        "file": "round2_prescreen_method_failures.csv",
        "meaning": "Optional method failures in round 2, if any.",
    },
    {
        "file": "round2_prescreen_recipe_failures.csv",
        "meaning": "Skipped prescreen recipes in round 2, if any.",
    },
    {"file": "stage_status_summary.json", "meaning": "Compact status summary."},
])
output_manifest.to_csv(OUTPUT_FILES["output_manifest"], index=False)
output_manifest

,file,meaning
0,stage_config.json,Alternative stage-02 execution configuration.
1,inner_cv_run_log.csv,Timestamped progress log.
2,inner_validation_cost_scale.csv,Per-inner-fold row-count diagnostic.
3,round1_inner_oof_scores.csv,Round 1 recipe scores computed directly from c...
4,round1_inner_oof_baselines.csv,Round 1 select-all and random baseline diagnos...
5,round1_selected_feature_recipes.csv,Feature recipes selected per outer fold for ro...
6,round2_inner_oof_scores.csv,Round 2 full pipeline scores computed directly...
7,round2_inner_oof_baselines.csv,Round 2 select-all and random baseline diagnos...
8,outer_selected_pipeline_recipes.csv,Final per-assessment-fold recipes for alternat...
9,round1_prescreen_method_failures.csv,"Optional method failures in round 1, if any."
